# 02 バックテスト & 最適化

## 目的
- 数万パターンのスクリーニング条件を自動生成
- ベクトル化バックテストで高速評価
- ランダムサーチ → ベイズ最適化 → 遺伝的アルゴリズムで最適条件発見
- Out-of-Sample検証で過学習チェック

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
%matplotlib inline

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')

In [ ]:
# 特徴量付きデータを読み込み
df = pd.read_pickle('../quant_research/data/_intermediate_df_features.pkl')
print(f"データ: {df['Code'].nunique():,} 銘柄, {len(df):,} 行")
print(f"期間: {df['Date'].min().date()} ~ {df['Date'].max().date()}")

## STEP 3: Train/Test 分割

In [ ]:
from quant_research.backtester import split_train_test

train_df, test_df = split_train_test(df, train_ratio=0.7)
print(f"\nTrain: {train_df['Date'].min().date()} ~ {train_df['Date'].max().date()} ({len(train_df):,} rows)")
print(f"Test:  {test_df['Date'].min().date()} ~ {test_df['Date'].max().date()} ({len(test_df):,} rows)")

## STEP 4: ランダムサーチ（広域探索）

In [ ]:
from quant_research.optimizer import random_search

# 広域探索（5000パターン。フル実行は50000推奨）
random_results = random_search(train_df, n_trials=5000, seed=42)

print(f"\n有効結果数: {len(random_results):,}")
print(f"\nTop 10:")
for i, r in enumerate(random_results[:10]):
    from quant_research.screener import condition_to_str
    print(f"  {i+1}. Score={r.composite_score:.4f} WR={r.win_rate:.1%} "
          f"Ret={r.avg_return:.2%} SR={r.sharpe_ratio:.2f} "
          f"N={r.n_trades} | {condition_to_str(r.condition)}")

## STEP 5: ベイズ最適化（深掘り）

In [ ]:
from quant_research.optimizer import bayesian_optimization

# ランダムサーチの上位結果をシードにベイズ最適化
bayesian_results, study = bayesian_optimization(
    train_df, 
    n_trials=500,   # フル実行は2000推奨
    seed_results=random_results[:100],
)

print(f"\nベイズ最適化 有効結果: {len(bayesian_results):,}")
print(f"Best score: {study.best_value:.4f}")
if bayesian_results:
    best = bayesian_results[0]
    print(f"Best: WR={best.win_rate:.1%} Ret={best.avg_return:.2%} SR={best.sharpe_ratio:.2f}")

## 遺伝的アルゴリズム（進化的発見）

In [ ]:
from quant_research.optimizer import genetic_algorithm

# 上位結果をシードにGA
top_seeds = sorted(
    random_results + bayesian_results,
    key=lambda r: r.composite_score, reverse=True
)[:50]

ga_results = genetic_algorithm(
    train_df,
    population_size=100,  # フル実行は200推奨
    n_generations=30,     # フル実行は100推奨
    seed_results=top_seeds,
)

print(f"\nGA 有効結果: {len(ga_results):,}")
if ga_results:
    best = ga_results[0]
    print(f"Best: WR={best.win_rate:.1%} Ret={best.avg_return:.2%} SR={best.sharpe_ratio:.2f}")

## Out-of-Sample 検証

In [ ]:
from quant_research.backtester import evaluate_out_of_sample

all_train = sorted(
    random_results + bayesian_results + ga_results,
    key=lambda r: r.composite_score, reverse=True
)

oos = evaluate_out_of_sample(all_train, test_df, top_n=30)

print(f"\n=== Out-of-Sample 検証結果 (Top 10) ===")
for i, o in enumerate(oos[:10]):
    decay_pct = o['score_decay'] * 100
    print(f"  {i+1}. Train: WR={o['train']['win_rate']:.1%} SR={o['train']['sharpe_ratio']:.2f} | "
          f"Test: WR={o['test']['win_rate']:.1%} SR={o['test']['sharpe_ratio']:.2f} | "
          f"Decay={decay_pct:+.1f}%")

In [ ]:
# 最適化結果の可視化
from quant_research.reporter import plot_equity_curves, plot_optimization_landscape

# Top 3 エクイティカーブ
top3 = all_train[:3]
plot_equity_curves(
    top3, 
    labels=['1st', '2nd', '3rd'],
    save_path='../quant_research/data/reports/equity_curves.png'
)

# 最適化ランドスケープ
plot_optimization_landscape(
    all_train[:500],
    save_path='../quant_research/data/reports/optimization_landscape.png'
)

plt.show()

In [ ]:
# 結果を保存
import pickle

opt_results = {
    'random': random_results,
    'bayesian': bayesian_results,
    'ga': ga_results,
    'all': all_train,
    'optuna_study': study,
    'best': all_train[0] if all_train else None,
}

pd.to_pickle(opt_results, '../quant_research/data/_results_optimization_results.pkl')
pd.to_pickle(train_df, '../quant_research/data/_intermediate_train_df.pkl')
pd.to_pickle(test_df, '../quant_research/data/_intermediate_test_df.pkl')
pd.to_pickle(oos, '../quant_research/data/_results_oos_results.pkl')

print("保存完了")